In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
import numpy as np
from collections import Counter


In [2]:
# Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = resnet18(pretrained=True).to(device).eval()
data = torchvision.datasets.CIFAR10(root='./data', train=False, download=True).data / 255


# Calculate mean and standard deviations
mean = data.mean(axis=(0,1,2))
std = data.std(axis=(0,1,2))
print(f"Mean: {mean} STD: {std}")
# Output: Mean : [0.491 0.482 0.446] STD: [0.247 0.243 0.261]

transform = transforms.Compose([
    transforms.ToTensor(),
    # CIFAR-10 Statistics - first section is the RGB means and the second is standard deviation
    transforms.Normalize(mean, std)
])

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=1, shuffle=True)


/Users/celinalinnerblom/Library/Python/3.9/lib/python/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/celinalinnerblom/Library/Python/3.9/lib/python/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Mean: [0.49421428 0.48513139 0.45040909] STD: [0.24665252 0.24289226 0.26159238]


In [3]:
# Fitness function
def fitness(image, x, y, r, g, b, model, true_label, device):
    # Create a copy of the image and modify the RGB values of the pixel
    img_copy = image.clone()
    img_copy[0, int(y), int(x)] = r / 255.0
    img_copy[1, int(y), int(x)] = g / 255.0
    img_copy[2, int(y), int(x)] = b / 255.0

    # Disable gradient computation (save computer power) since we only need predictions
    with torch.no_grad():
        # Add batch dimension, send to GPU and pass it through ResNet-18
        output = model(img_copy.unsqueeze(0).to(device))
        # Normalize confidences using softmax
        confidences = torch.softmax(output, dim=1)[0]
        # Find the predicted class / class with highest probability
        pred = torch.argmax(output, dim=1).item()

    # Verify if attack was successful / prediction label equals true label
    if pred != true_label:
        # Attack was successful, return the confidence 
        return confidences[pred].item()
    else:
        # Attack failed / prediction label equals true label
        return 0.0

In [ ]:
def mutate(solution):
    # Make a copy of the solution
    s = solution.copy()
    
    # Mutate the coordinates with small random changes
    s[0] = int(np.clip(s[0] + np.random.normal(0, 15), 0, 31))
    s[1] = int(np.clip(s[1] + np.random.normal(0, 15), 0, 31))
    
    # Mutate the RGB values with larger random numbers
    for i in range(2, 5):
        s[i] = int(np.clip(s[i] + np.random.normal(0, 50), 0, 255))
        
    return s

In [5]:
def attack(image, true_label, model, device, gens=300):
    # Create a population with 15 random candidates (random coordinate + RGB values)
    pop = [np.array([np.random.randint(0,32), np.random.randint(0,32), 
                     np.random.randint(0,256), np.random.randint(0,256), 
                     np.random.randint(0,256)]) for _ in range(15)]

    # Variables to track the best solution
    best_ever = pop[0].copy()
    best_ever_fitness = 0
    gen_found = -1

    # Iterate through the generations
    for gen in range(gens):
        # Evaluate candidates in population and get best solution
        fitnesses = [fitness(image, s[0], s[1], s[2], s[3], s[4], model, true_label, device) for s in pop]
        best_idx = np.argmax(fitnesses)
        best_fit = fitnesses[best_idx]

        # If better solution is found, update the tracking variables
        if best_fit > best_ever_fitness:
            best_ever_fitness = best_fit
            best_ever = pop[best_idx].copy()

        # If attack is successful, stop the attack
        if best_fit > 0.5:
            gen_found = gen
            break

        # Create a new generation 
        new_pop = [pop[best_idx].copy()]
        for _ in range(14):
            new_pop.append(mutate(pop[best_idx].copy()))
        pop = new_pop

    # Verify if attack was successful and return best solution
    success = best_ever_fitness > 0.5
    return best_ever, success, gen_found

In [ ]:
# Test
print("Testing attacks...")
success_count = 0
gen_list = []
success_labels = []
tested_labels = []

for i, (img, label) in enumerate(testloader):
    if i >= 5:  # Amounts of attacks / images
        break

    # Remove batch dimension and get the label
    img = img[0]
    label = label.item()
    # Save tested labels
    tested_labels.append(label) 

    # Perform attack
    best, success, gen = attack(img, label, model, device, gens=300)

    # Increase success count if attack was successful
    if success:
        success_count += 1
        gen_list.append(gen)
        success_labels.append(label)
        print(f"[{i}] Success - Class: {label} - Gen: {gen}")

    else:
        print(f"[{i}] Failed - Class: {label}")

# Analyze successful attacks per class
label_counts = Counter(success_labels)
print("Successful attacks per claass:")
print(label_counts)

# Print final success rate and average generations
total_images = i + 1
success_rate = success_count / total_images * 100

print(f"\nSuccess rate: {success_rate:.2f}% ({success_count}/{total_images})")

print(f"\nSuccess rate: {success_count}/10")
if gen_list:
    print(f"Avg generations: {np.mean(gen_list):.1f}")

# Analyze vulnerability per class
tested_count = Counter(tested_labels)
success_count = Counter(success_labels)

print("\Vulnerability per class (success / tested):")
for cls in tested_count:
    rate = success_count[cls] / tested_count[cls]
    print(f"Class {cls}: {rate:.2f}")


Testing attacks...
Mutate: x=2, y=7, r=36, g=255, b=203
Mutate: x=21, y=27, r=0, g=172, b=178
Mutate: x=0, y=13, r=121, g=255, b=130
Mutate: x=31, y=28, r=0, g=236, b=167
Mutate: x=15, y=20, r=57, g=242, b=217
Mutate: x=11, y=20, r=19, g=255, b=91
Mutate: x=0, y=3, r=0, g=255, b=176
Mutate: x=0, y=7, r=4, g=220, b=80
Mutate: x=19, y=31, r=60, g=190, b=144
Mutate: x=31, y=29, r=44, g=175, b=144
Mutate: x=3, y=0, r=2, g=255, b=77
Mutate: x=0, y=21, r=25, g=255, b=130
Mutate: x=4, y=5, r=47, g=171, b=216
Mutate: x=26, y=0, r=53, g=255, b=201
Mutate: x=10, y=4, r=84, g=255, b=185
Mutate: x=0, y=26, r=0, g=187, b=255
Mutate: x=0, y=14, r=30, g=213, b=162
Mutate: x=20, y=21, r=36, g=168, b=255
Mutate: x=15, y=10, r=0, g=185, b=255
Mutate: x=31, y=3, r=5, g=198, b=209
Mutate: x=31, y=31, r=56, g=255, b=221
Mutate: x=31, y=8, r=117, g=155, b=212
Mutate: x=20, y=27, r=28, g=190, b=196
Mutate: x=31, y=9, r=32, g=255, b=136
Mutate: x=23, y=29, r=104, g=201, b=226
Mutate: x=7, y=10, r=30, g=255, b